[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/01_python_for_ml/03_numpy_vectorization_and_broadcasting.ipynb)

# Bài 03 — Vectorization & Broadcasting: vì sao `X @ W` thay cho `for`

**🎯 Mục tiêu**
- Hiểu vì sao code vectorized nhanh hơn vòng lặp Python hàng chục–hàng trăm lần.
- Nắm **quy tắc broadcasting** và dùng nó để chuẩn hoá dữ liệu, tính ma trận khoảng cách, softmax.
- Viết dự đoán, loss và gradient của một model tuyến tính **không có vòng lặp** — chuẩn bị cho chương 06.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/01_python_for_ml

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

from ml_course.visualization import set_style

set_style()
rng = np.random.default_rng(0)

## 1. Vòng lặp vs vectorization

Dự đoán của một model tuyến tính cho **một** mẫu là tích vô hướng $\hat y = \mathbf{w}^\top \mathbf{x} = \sum_j w_j x_j$. Với $n$ mẫu xếp thành ma trận $X$ (shape `(n, d)`), toàn bộ dự đoán là **một phép nhân ma trận**:

$$\hat{\mathbf{y}} = X\mathbf{w}$$

So sánh 3 cách tính:

In [ ]:
n, d = 20_000, 50
X = rng.normal(size=(n, d))
w = rng.normal(size=d)


def predict_double_loop(X, w):
    y = []
    for i in range(X.shape[0]):
        s = 0.0
        for j in range(X.shape[1]):
            s += X[i, j] * w[j]
        y.append(s)
    return np.array(y)


def predict_single_loop(X, w):
    return np.array([np.dot(x_i, w) for x_i in X])


def predict_vectorized(X, w):
    return X @ w


for fn in (predict_double_loop, predict_single_loop, predict_vectorized):
    start = time.perf_counter()
    y_hat = fn(X, w)
    print(f"{fn.__name__:22s} {1000 * (time.perf_counter() - start):9.2f} ms")

assert np.allclose(predict_double_loop(X[:100], w), predict_vectorized(X[:100], w))

**Vì sao nhanh hơn?**
1. Vòng lặp Python phải kiểm tra kiểu dữ liệu, cấp phát object cho *từng* phép nhân. NumPy chạy vòng lặp trong **C** trên mảng liên tục trong bộ nhớ.
2. Phép nhân ma trận dùng thư viện **BLAS** tối ưu theo CPU: lệnh SIMD (nhiều phép tính trong một lệnh), cache, đa luồng.
3. Trên GPU (PyTorch), cùng một biểu thức `X @ W` chạy song song trên hàng nghìn lõi.

> **Quy tắc thực hành:** trong code ML, nếu bạn thấy vòng `for` duyệt qua *từng mẫu* hoặc *từng feature*, hãy tự hỏi: có viết được bằng phép toán ma trận không?

Thời gian chạy theo kích thước dữ liệu:

In [ ]:
sizes = [100, 300, 1000, 3000, 10000]
t_loop, t_vec = [], []
for m in sizes:
    Xm = X[:m]
    start = time.perf_counter(); predict_single_loop(Xm, w); t_loop.append(time.perf_counter() - start)
    start = time.perf_counter(); predict_vectorized(Xm, w); t_vec.append(time.perf_counter() - start)

fig, ax = plt.subplots()
ax.loglog(sizes, t_loop, "o-", label="vòng lặp Python")
ax.loglog(sizes, t_vec, "s-", label="X @ w (vectorized)")
ax.set_xlabel("số mẫu n")
ax.set_ylabel("thời gian (s)")
ax.set_title("Vectorization nhanh hơn ở mọi kích thước")
ax.legend()
plt.show()

## 2. Quy tắc broadcasting

Khi thực hiện phép toán giữa hai mảng khác shape, NumPy **so sánh shape từ phải sang trái**. Hai chiều tương thích nếu:
- chúng **bằng nhau**, hoặc
- một trong hai **bằng 1** (chiều đó sẽ được "kéo giãn").

Nếu một mảng có ít chiều hơn, NumPy thêm chiều 1 vào **bên trái**.

| A | B | Kết quả |
|---|---|---|
| `(4, 3)` | `(3,)` | `(4, 3)` — `(3,)` được coi như `(1, 3)` |
| `(4, 3)` | `(4, 1)` | `(4, 3)` |
| `(4, 1)` | `(1, 5)` | `(4, 5)` — "outer" |
| `(4, 3)` | `(4,)` | ❌ lỗi: 3 ≠ 4 |
| `(n, 1, d)` | `(1, m, d)` | `(n, m, d)` |

In [ ]:
A = np.ones((4, 3))
print((A + np.arange(3)).shape)               # (4, 3)
print((A + np.arange(4).reshape(4, 1)).shape) # (4, 3)
print((np.arange(4)[:, None] * np.arange(5)[None, :]).shape)  # (4, 5) — bảng cửu chương thu nhỏ
try:
    A + np.arange(4)
except ValueError as e:
    print("ValueError:", e)

`x[:, None]` (hay `x[:, np.newaxis]`) thêm một chiều độ dài 1 — công cụ chính để "sắp" shape cho broadcasting.

## 3. Ứng dụng 1 — Chuẩn hoá dữ liệu (standardization)

$$z_{ij} = \frac{x_{ij} - \mu_j}{\sigma_j}$$

trong đó $\mu_j, \sigma_j$ là trung bình và độ lệch chuẩn của **feature** $j$.

In [ ]:
X_raw = np.column_stack([
    rng.normal(70, 20, 1000),     # diện tích (m²)
    rng.normal(3, 1, 1000),       # số phòng
    rng.normal(5e9, 1e9, 1000),   # giá (VND) — thang đo rất khác!
])
mu = X_raw.mean(axis=0)           # shape (3,)
sigma = X_raw.std(axis=0)         # shape (3,)
Z = (X_raw - mu) / sigma          # (1000, 3) - (3,) → broadcasting theo hàng

print("mean sau chuẩn hoá:", Z.mean(axis=0).round(6))
print("std sau chuẩn hoá :", Z.std(axis=0).round(6))

## 4. Ứng dụng 2 — Ma trận khoảng cách (nền tảng của KNN, K-Means)

Cần khoảng cách Euclid giữa mọi cặp điểm $a_i$ (từ `A`, shape `(n, d)`) và $b_j$ (từ `B`, shape `(m, d)`).

**Cách 1 — broadcasting trực tiếp:** `A[:, None, :] - B[None, :, :]` có shape `(n, m, d)`.

**Cách 2 — khai triển:** $\lVert a - b\rVert^2 = \lVert a\rVert^2 + \lVert b\rVert^2 - 2a^\top b$ → chỉ cần một phép nhân ma trận `A @ B.T`, tiết kiệm bộ nhớ hơn nhiều.

In [ ]:
A = rng.normal(size=(500, 20))
B = rng.normal(size=(300, 20))


def dist_loops(A, B):
    D = np.zeros((len(A), len(B)))
    for i in range(len(A)):
        for j in range(len(B)):
            D[i, j] = np.sqrt(np.sum((A[i] - B[j]) ** 2))
    return D


def dist_broadcast(A, B):
    return np.sqrt(((A[:, None, :] - B[None, :, :]) ** 2).sum(axis=-1))


def dist_expand(A, B):
    sq = (A**2).sum(1)[:, None] + (B**2).sum(1)[None, :] - 2 * A @ B.T
    return np.sqrt(np.maximum(sq, 0))   # maximum: tránh số âm rất nhỏ do sai số làm tròn


for fn in (dist_loops, dist_broadcast, dist_expand):
    start = time.perf_counter()
    D = fn(A, B)
    print(f"{fn.__name__:15s} {1000 * (time.perf_counter() - start):8.1f} ms  shape={D.shape}")

print("Kết quả giống nhau:", np.allclose(dist_loops(A[:20], B[:20]), dist_expand(A[:20], B[:20])))

⚠️ **Broadcasting không miễn phí về bộ nhớ:** cách 1 tạo mảng trung gian `(n, m, d)`. Với n = m = 10 000 và d = 100 bằng float64, đó là 80 GB! Cách 2 chỉ cần `(n, m)`.

## 5. Ứng dụng 3 — Softmax theo hàng (`keepdims`)

$$\text{softmax}(\mathbf{z})_k = \frac{e^{z_k}}{\sum_j e^{z_j}}$$

`keepdims=True` giữ chiều đã gộp với độ dài 1, để kết quả broadcast đúng với mảng ban đầu:

In [ ]:
logits = np.array([[2.0, 1.0, 0.1], [1000.0, 1000.0, 999.0]])

s = logits.sum(axis=1)
print("không keepdims:", s.shape, "| keepdims:", logits.sum(axis=1, keepdims=True).shape)


def softmax(z):
    z = z - z.max(axis=1, keepdims=True)   # trừ max để exp không bị overflow (ổn định số học)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


P = softmax(logits)
print(P.round(4))
print("tổng mỗi hàng:", P.sum(axis=1))

Thử bỏ dòng trừ `max`: `np.exp(1000)` = `inf` → kết quả `nan`. Đây là ví dụ đầu tiên về **ổn định số học** — một chủ đề quay lại nhiều lần trong khoá học.

## 6. Ứng dụng 4 — Model tuyến tính hoàn toàn vectorized

Đây là "bộ khung" của Linear Regression ở chương 06. Mọi thứ — dự đoán, loss, gradient — đều là phép toán ma trận:

$$\hat{\mathbf y} = X\mathbf w + b, \qquad L = \frac{1}{n}\lVert \hat{\mathbf y} - \mathbf y\rVert^2, \qquad \nabla_{\mathbf w} L = \frac{2}{n}X^\top(\hat{\mathbf y} - \mathbf y)$$

In [ ]:
n, d = 1000, 3
X = rng.normal(size=(n, d))
true_w, true_b = np.array([2.0, -1.0, 0.5]), 4.0
y = X @ true_w + true_b + rng.normal(0, 0.1, n)

w, b, lr = np.zeros(d), 0.0, 0.1
for step in range(200):
    y_hat = X @ w + b                       # (n,)
    err = y_hat - y                         # (n,)
    loss = np.mean(err**2)
    grad_w = 2 / n * X.T @ err              # (d,)  — không vòng lặp qua mẫu!
    grad_b = 2 * err.mean()
    w -= lr * grad_w
    b -= lr * grad_b

print("w học được:", w.round(3), "| w thật:", true_w)
print("b học được:", round(b, 3), "| b thật:", true_b)
print("loss cuối :", round(loss, 5))

## 🧠 Kiểm tra nhanh

1. Shape `(5, 1)` và `(1, 4)` broadcast thành shape gì? `(5, 3)` và `(5,)` thì sao?
2. Vì sao cần `keepdims=True` khi chia `e / e.sum(axis=1, keepdims=True)`?
3. Tính ma trận khoảng cách giữa 10 000 và 10 000 điểm 100 chiều — cách nào an toàn về bộ nhớ?
4. Viết gradient $\frac{2}{n}X^\top(\hat y - y)$ bằng vòng lặp qua từng mẫu. Vì sao phiên bản ma trận tương đương?

## 📌 Tóm tắt

- Vectorization = đẩy vòng lặp xuống C/BLAS (CPU) hoặc CUDA (GPU) → nhanh hơn hàng trăm lần.
- Broadcasting so shape từ phải sang trái; chiều bằng 1 được kéo giãn; dùng `[:, None]` để thêm chiều.
- Cẩn thận mảng trung gian khổng lồ khi broadcast 3 chiều.
- Dự đoán, loss, gradient của model tuyến tính đều viết được bằng vài phép toán ma trận.

## 📚 Tài liệu tham khảo

- NumPy — Broadcasting: https://numpy.org/doc/stable/user/basics.broadcasting.html
- *Deep Learning Specialization* (Coursera) — Vectorization
- Bài tập phần vectorization trong `93_assignment_python_basics_with_numpy.ipynb` (từ khoá AIVN)